# Feature Selection using without MRMR - All Base data

In [7]:
import core.constants as c
from core.utils import save_df_as_table_image, extract_subject_id
import os

import warnings

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from feature_engine.selection import SmartCorrelatedSelection
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedGroupKFold

from feature_engine.selection import MRMR

import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
SAVE_DATA = False

In [9]:
# Base features no extensive outlier treatment.
data = pd.read_csv(c.RICKD_FEATURE_SELECTION_DATA_FILE_2, index_col=0)
display(data.head())

data.info(verbose=True)

,speed_output,step_width_left,step_width_right,stride_rate_left,stride_rate_right,stride_length_left,stride_length_right,swing_time_left,swing_time_right,stance_time_left,...,pelvic_drop_peak_vel_left,pelvic_drop_peak_vel_right,vertical_oscillation_left,vertical_oscillation_right,age,height,weight,gender,dominantleg,is_injured
id,,,,,,,,,,,,,,,,,,,,,
100001_20110531T161051,2.489233,0.123419,0.123419,78.947368,78.947368,1.891817,1.891817,0.4350,0.420,0.320,...,-82.090017,-57.417845,96.433793,92.586134,47.0,172.0,61.9,female,left,True
100002_20110601T140505,2.722687,0.032922,0.032922,81.632653,81.632653,2.001175,2.001175,0.4450,0.420,0.290,...,-57.724685,-60.462411,86.521432,94.945518,37.0,173.4,70.6,male,left,True
100003_20110601T095930,2.949904,0.097273,0.097273,78.947368,78.947368,2.241927,2.241927,0.4475,0.445,0.315,...,-82.171073,-95.302664,82.680986,76.611379,51.0,186.0,86.5,male,right,True
100004_20110203T120721,2.688014,0.011401,0.011401,82.191781,82.191781,1.962250,1.962250,0.4400,0.460,0.290,...,-63.319119,-49.646132,92.593339,83.273183,35.0,175.6,59.0,male,left,True
100004_20140929T102035,2.928598,0.029021,0.029021,83.333333,82.758621,2.108590,2.123233,0.4200,0.430,0.300,...,-32.717949,-52.004473,87.481400,75.091789,39.0,175.0,61.0,male,left,False


<class 'pandas.core.frame.DataFrame'>
Index: 1813 entries, 100001_20110531T161051 to 201225_20140515T133244
Data columns (total 87 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   speed_output                    1813 non-null   float64
 1   step_width_left                 1813 non-null   float64
 2   step_width_right                1813 non-null   float64
 3   stride_rate_left                1813 non-null   float64
 4   stride_rate_right               1813 non-null   float64
 5   stride_length_left              1813 non-null   float64
 6   stride_length_right             1813 non-null   float64
 7   swing_time_left                 1813 non-null   float64
 8   swing_time_right                1813 non-null   float64
 9   stance_time_left                1813 non-null   float64
 10  stance_time_right               1813 non-null   float64
 11  pelvis_peak_drop_angle_left     1813 non-null   float64
 12  

In [10]:
from core.evaluation import standardise_and_split
from core.utils import extract_subject_id

y_label = "is_injured"
y = data[y_label]
X = data.drop(columns=[y_label])

feature_categorical_columns = ["gender", "dominantleg"]

(result, pipeline) = standardise_and_split(
    X, y, groups=extract_subject_id(X.index.to_series()),
    feature_categorical_columns=feature_categorical_columns,
    test_size=0.2, val_size=0.1,
    return_pipeline=True,
)

(X_train, X_val, X_test,
 y_train, y_val, y_test,
 groups_train, groups_val, groups_test) = result

(imputer, preprocessing) = pipeline

print("X_train.shape: ", X_train.shape)
print("y_train.shape: ", y_train.shape)
print("X_val.shape: ", X_val.shape)
print("y_val.shape: ", y_val.shape)
print("X_test.shape: ", X_test.shape)
print("y_test.shape: ", y_test.shape)
print("groups_train.shape: ", groups_train.shape)
print("groups_val.shape: ", groups_val.shape)
print("groups_test.shape: ", groups_test.shape)
print(50*"=")
print("Class distribution: Train:", y_train.value_counts().to_dict(), "Percentage:", (y_train.value_counts() / y_train.shape[0] * 100).to_dict())
print("Class distribution: Val:", y_val.value_counts().to_dict(), "Percentage:", (y_val.value_counts() / y_val.shape[0] * 100).to_dict())
print("Class distribution: Test:", y_test.value_counts().to_dict(), "Percentage:", (y_test.value_counts() / y_test.shape[0] * 100).to_dict())

X_train.shape:  (1254, 89)
y_train.shape:  (1254,)
X_val.shape:  (192, 89)
y_val.shape:  (192,)
X_test.shape:  (367, 89)
y_test.shape:  (367,)
groups_train.shape:  (1254,)
groups_val.shape:  (192,)
groups_test.shape:  (367,)
Class distribution: Train: {True: 803, False: 451} Percentage: {True: 64.03508771929825, False: 35.96491228070175}
Class distribution: Val: {True: 124, False: 68} Percentage: {True: 64.58333333333334, False: 35.41666666666667}
Class distribution: Test: {True: 224, False: 143} Percentage: {True: 61.03542234332425, False: 38.96457765667575}


In [ ]:
# Save
if SAVE_DATA:
    print("Saving data...")
    X_train.to_csv(c.RICKD_MODEL_1_X_TRAIN_BASE_STANDARDISED)
    y_train.to_csv(c.RICKD_MODEL_1_Y_TRAIN_BASE_STANDARDISED)

    X_test.to_csv(c.RICKD_MODEL_1_X_TEST_BASE_STANDARDISED)
    y_test.to_csv(c.RICKD_MODEL_1_Y_TEST_BASE_STANDARDISED)
    
    X_val.to_csv(c.RICKD_MODEL_1_X_VAL_BASE_STANDARDISED)
    y_val.to_csv(c.RICKD_MODEL_1_Y_VAL_BASE_STANDARDISED)

In [12]:
# Read
if not SAVE_DATA:
    X_train = pd.read_csv(c.RICKD_MODEL_1_X_TRAIN_BASE_STANDARDISED, index_col=0)
    y_train = pd.read_csv(c.RICKD_MODEL_1_Y_TRAIN_BASE_STANDARDISED, index_col=0)

    X_test = pd.read_csv(c.RICKD_MODEL_1_X_TEST_BASE_STANDARDISED, index_col=0)
    y_test = pd.read_csv(c.RICKD_MODEL_1_Y_TEST_BASE_STANDARDISED, index_col=0)

    X_val = pd.read_csv(c.RICKD_MODEL_1_X_VAL_BASE_STANDARDISED, index_col=0)
    y_val = pd.read_csv(c.RICKD_MODEL_1_Y_VAL_BASE_STANDARDISED, index_col=0)